In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scikit_posthocs as sp
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import scipy.stats as ss
import seaborn as sns
from tox_ames.utils import plot_confusion_matrix_from_cv, plot_mean_roc
from tox_ames.utils.style import (
    get_model_style,
    get_manuscript_label,
    get_dataset_label,
    MODEL_STYLES,
)
from pathlib import Path
import scienceplots

plt.style.use("science")

DATASET = "combined"
SHOW_ENSEMBLE = True
BASE_DIR = Path(f"../notebooks/logs/{DATASET}")
FIG_DIR = Path(f"Figs/results/{DATASET}")
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_2_MODELS = [
    "exp-fp_ecfp-scf",
    "exp-fp_avalon-scf",
    "exp-fp_maccs-scf",
    "exp-fp_pubchem-scf",
    "exp-chemberta_77M-MLM_mean-scf",
    "exp-chemberta_77M-MTR_mean-scf",
    "exp-mpnn_mpnn_atom_norm-scf",
    "exp-mpnn_mpnn_norm-scf",
    "exp-unimolv1_ft-scf",
    "exp-unimolv2_ft-scf",
]
model_files = {
    name: BASE_DIR / name / "cv_predictions.csv"
    for name in TABLE_2_MODELS
    if (BASE_DIR / name / "cv_predictions.csv").exists()
}
ensemble_file = BASE_DIR / "advanced_analysis" / "ensemble_predictions.csv"
if SHOW_ENSEMBLE and ensemble_file.exists():
    model_files["Ensemble"] = ensemble_file


def publication_label(name):
    # Short manuscript labels, identical to Table 3, so every figure entry
    # maps unambiguously to a table row.
    return "Cross-repr." if name == "Ensemble" else get_manuscript_label(name)


In [ ]:
# =====================================================================
# 1. EXTRACT SCORES INTO A LONG DATAFRAME
# =====================================================================
rows = []
for name, path in model_files.items():
    df = pd.read_csv(path)
    for fold in sorted(df["fold"].unique()):
        fold_data = df[df["fold"] == fold]
        score_auc = roc_auc_score(fold_data["y_actual"], fold_data["y_prob"])
        score_acc = accuracy_score(fold_data["y_actual"], fold_data["y_pred"])
        score_f1 = f1_score(fold_data["y_actual"], fold_data["y_pred"])

        rows.append({"Model": name, "Fold": fold, "Score": score_auc, "Metric": "AUC"})
df_long = pd.DataFrame(rows)


In [ ]:
# Map model names to publication labels for plotting and tests
df_long_mapped = df_long.copy()
df_long_mapped["Model"] = df_long_mapped["Model"].map(publication_label)


In [ ]:
# Generate the palette in the same order used by the manuscript results table
palette = {}
ordered_model_names = []
available_raw_names = set(df_long["Model"].unique())

for raw_name in TABLE_2_MODELS:
    if raw_name in available_raw_names:
        mapped_name = publication_label(raw_name)
        palette[mapped_name] = get_model_style(raw_name).color
        ordered_model_names.append(mapped_name)

for raw_name in sorted(available_raw_names - set(TABLE_2_MODELS)):
    mapped_name = publication_label(raw_name)
    palette[mapped_name] = get_model_style(raw_name).color
    ordered_model_names.append(mapped_name)


In [ ]:
# =====================================================================
# 3. RUN THE CONOVER-FRIEDMAN POST-HOC TEST
# =====================================================================
model_order = [
    name for name in ordered_model_names if name in set(df_long_mapped["Model"])
]
score_matrix = (
    df_long_mapped.pivot(index="Fold", columns="Model", values="Score")
    .sort_index()
    .loc[:, model_order]
)
if score_matrix.isna().any().any():
    raise ValueError("Every model must have one ROC-AUC value for every holdout.")

# Normalized ranks follow the main-text convention: higher is better.
avg_rank = score_matrix.rank(axis=1, pct=True).mean(axis=0)
p_matrix = sp.posthoc_conover_friedman(
    df_long_mapped,
    y_col="Score",
    group_col="Model",
    block_col="Fold",
    block_id_col="Fold",
    melted=True,
    p_adjust="holm",
)

p_matrix = p_matrix.loc[model_order, model_order]
friedman_statistic, friedman_p_value = ss.friedmanchisquare(
    *(score_matrix[name].to_numpy() for name in model_order)
)
print(f"Friedman statistic: {friedman_statistic:.4f}, p-value: {friedman_p_value:.4e}")

# =====================================================================
# 4. SAVE PUBLICATION-READY SUPPLEMENTARY FIGURES
# =====================================================================
heatmap_dir = FIG_DIR / "friedman_conover"
rank_dir = FIG_DIR / "critical_difference"
heatmap_dir.mkdir(parents=True, exist_ok=True)
rank_dir.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(7.0, 6.0), dpi=100)
ax.set_title(f"Friedman omnibus: p = {friedman_p_value:.2e}", pad=12)
# Format: diagonal, non-significant, p<0.001, p<0.01, p<0.05
cmap = ["1", "#fb6a4a", "#08306b", "#4292c6", "#c6dbef"]
heatmap_args = {
    "cmap": cmap,
    "linewidths": 0.35,
    "linecolor": "0.65",
    "ax": ax,
    "cbar_ax_bbox": [0.91, 0.35, 0.025, 0.30],
}
sp.sign_plot(p_matrix, **heatmap_args)
ax.set_xticklabels(
    ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor"
)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.tick_params(axis="both", labelsize=9)
fig.subplots_adjust(left=0.20, right=0.87, bottom=0.22, top=0.89)
fig.savefig(heatmap_dir / "plot.png", dpi=300, bbox_inches="tight", pad_inches=0.05)
plt.show()

fig, ax = plt.subplots(figsize=(8.5, 4.3), dpi=100)
artists = sp.critical_difference_diagram(
    avg_rank,
    p_matrix,
    ax=ax,
    color_palette=palette,
    label_fmt_left="{label} ({rank:.2f})",
    label_fmt_right="({rank:.2f}) {label}",
    label_props={"fontsize": 9},
    marker_props={"s": 28, "edgecolor": "black", "linewidth": 0.4},
    elbow_props={"linewidth": 1.0},
    crossbar_props={"linewidth": 1.6},
)
for label in artists["labels"]:
    label.set_color("black")
ax.set_title(
    "Average normalized within-holdout ROC-AUC ranks (higher is better)", pad=14
)
fig.subplots_adjust(left=0.03, right=0.97, bottom=0.04, top=0.84)
fig.savefig(rank_dir / "plot.png", dpi=300, bbox_inches="tight", pad_inches=0.05)
plt.show()


In [ ]:
# plot with seaborn and stripplot
plt.figure(figsize=(7.5, 5.7))
# plt.title(f"Model Performance Comparison ({get_dataset_label(BASE_DIR.name)})")

# Plot strip points for each model using its custom marker and color from datamodel
for raw_name in df_long["Model"].unique():
    mapped_name = publication_label(raw_name)
    style = get_model_style(raw_name)
    model_data = df_long_mapped[df_long_mapped["Model"] == mapped_name]
    sns.stripplot(
        x="Model",
        y="Score",
        data=model_data,
        order=ordered_model_names,
        jitter=0.18,
        edgecolor="k",
        linewidth=0.3,
        color=style.color,
        size=4,
        alpha=0.25,
    )

sns.boxplot(
    x="Model",
    y="Score",
    data=df_long_mapped,
    order=ordered_model_names,
    whis=np.inf,
    showfliers=False,
    palette=palette,
    hue="Model",
    dodge=False,
)
sns.despine(right=False, top=False)

# make xlabel 30 degree
plt.xticks(rotation=30, ha="right")
plt.grid(linestyle=":")
plt.tight_layout()
plt.ylabel("Test ROC-AUC")
plt.ylim(0.75, 0.95)
plt.savefig(FIG_DIR / "model_performance_comparison.png", bbox_inches="tight", dpi=600)
plt.show()


In [ ]:
# Map model folder names to publication labels
df_long_mapped = df_long.copy()
df_long_mapped["Model"] = df_long_mapped["Model"].map(publication_label)

# Run posthoc again on mapped names
p_matrix_mapped = sp.posthoc_conover_friedman(
    df_long_mapped,
    y_col="Score",
    group_col="Model",
    block_col="Fold",
    block_id_col="Fold",
    melted=True,
    p_adjust="holm",
)

summary = df_long_mapped.groupby("Model")["Score"].agg(["mean", "std"]).reset_index()
summary["se"] = 1.96 * (summary["std"] / np.sqrt(10))

# Order by the clean family-based order
summary["Model"] = pd.Categorical(
    summary["Model"], categories=ordered_model_names, ordered=True
)
summary = summary.sort_values(by="Model").reset_index(drop=True)
summary.dropna(inplace=True)
summary
# Dynamically check which mapped model name to use as reference
ref_models = [m for m in summary["Model"].unique() if "Uni-Mol v2" in str(m)]
REFERENCE_MODEL = ref_models[0] if ref_models else summary["Model"].unique()[0]


In [ ]:
colors = []
for name in summary["Model"]:
    if name == REFERENCE_MODEL:
        colors.append("salmon")
    else:
        p_val = p_matrix_mapped.loc[name, REFERENCE_MODEL]
        colors.append("dodgerblue" if p_val < 0.05 else "lightgray")

plt.figure(figsize=(7.5, 5.7), dpi=300)
plt.title(
    f"Pairwise Comparison Protocol (Dataset: {get_dataset_label(BASE_DIR.name)}, Reference: {REFERENCE_MODEL})",
    fontweight="bold",
    pad=15,
)

plt.hlines(
    y=summary["Model"],
    xmin=summary["mean"] - summary["se"],
    xmax=summary["mean"] + summary["se"],
    colors=colors,
    linewidth=2.5,
)
plt.scatter(summary["mean"], summary["Model"], color=colors, s=60, zorder=3)

ref_idx = list(summary["Model"]).index(REFERENCE_MODEL)
plt.axvline(
    x=summary["mean"].iloc[ref_idx] - summary["se"].iloc[ref_idx],
    color="gray",
    linestyle="--",
)
plt.axvline(
    x=summary["mean"].iloc[ref_idx] + summary["se"].iloc[ref_idx],
    color="gray",
    linestyle="--",
)
plt.xlabel("Test ROC-AUC")
plt.gca().spines[["top", "right"]].set_visible(False)
sns.despine(right=False, top=False)
plt.grid(linestyle=":")
plt.tight_layout()
plt.savefig(FIG_DIR / "pairwise_comparison_protocol.png", bbox_inches="tight", dpi=600)
plt.show()


In [ ]:
# Mean normalized within-holdout ROC-AUC ranks
from IPython.display import Image, display
from tox_ames.utils.plot_normalized_auc_ranks import (
    ENSEMBLE_OUTPUT_PATH,
    OUTPUT_PATH,
    plot_normalized_auc_ranks,
)

plot_normalized_auc_ranks(include_ensemble=SHOW_ENSEMBLE)
rank_output = ENSEMBLE_OUTPUT_PATH if SHOW_ENSEMBLE else OUTPUT_PATH
display(Image(filename=rank_output))


In [ ]:
# Mean normalized within-holdout F1 ranks
from tox_ames.utils.plot_normalized_auc_ranks import (
    F1_ENSEMBLE_OUTPUT_PATH,
    F1_OUTPUT_PATH,
)

plot_normalized_auc_ranks(include_ensemble=SHOW_ENSEMBLE, metric="f1")
f1_rank_output = F1_ENSEMBLE_OUTPUT_PATH if SHOW_ENSEMBLE else F1_OUTPUT_PATH
display(Image(filename=f1_rank_output))


In [ ]:
# Plot confusion matrix and ROC curves for each experiment dynamically
SAVE_DIR = Path(f"Figs/results/{DATASET}/cm_and_roc")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
for exp in model_files.keys():
    prediction_df = pd.read_csv(model_files[exp])
    style = get_model_style(exp)
    model_label = publication_label(exp)

    plot_confusion_matrix_from_cv(
        prediction_df,
        fold=None,
        title=f"Confusion Matrix - {model_label} ({get_dataset_label(BASE_DIR.name)})",
        cmap=style.cmap,  # fallback to color
        save_path=SAVE_DIR / f"confusion_matrix_{model_label.replace(' ', '_')}.png",
    )
    plot_mean_roc(
        prediction_df,
        color=style.color,
        title=f"ROC with Std (Across Folds) for model {model_label} ({get_dataset_label(BASE_DIR.name)})",
        save_path=SAVE_DIR / f"roc_curve_{model_label.replace(' ', '_')}.png",
    )


In [ ]:
# Combined-dataset AUROC summary for the evaluated pipelines
auroc_summary = (
    df_long_mapped.groupby("Model")["Score"]
    .agg(mean="mean", std="std")
    .mul(100)
    .sort_values("mean")
)
auroc_dir = FIG_DIR / "auroc_summary"
auroc_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(7.0, 4.8))
for y_position, (model, row) in enumerate(auroc_summary.iterrows()):
    color = palette[model]
    ax.errorbar(
        row["mean"],
        y_position,
        xerr=row["std"],
        fmt="o",
        color=color,
        ecolor=color,
        markeredgecolor="black",
        markeredgewidth=0.6,
        markersize=7,
        elinewidth=2,
        capsize=3,
        capthick=1.2,
        zorder=3,
    )
    ax.text(
        row["mean"] + row["std"] + 0.3,
        y_position,
        f"{row['mean']:.2f}",
        va="center",
        fontsize=8.5,
    )

lower = (auroc_summary["mean"] - auroc_summary["std"]).min() - 0.8
upper = (auroc_summary["mean"] + auroc_summary["std"]).max() + 1.5
ax.set_xlim(lower, upper)
ax.set_yticks(range(len(auroc_summary)), labels=auroc_summary.index)
ax.set_xlabel("Test ROC-AUC (percentage points)")
ax.set_ylabel("")
ax.set_title("Combined-dataset AUROC across evaluated pipelines")
ax.grid(axis="x", linestyle=":", linewidth=0.6, alpha=0.7)
ax.grid(axis="y", visible=False)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(direction="out", length=4)
ax.tick_params(axis="y", length=0)
fig.tight_layout()
fig.savefig(auroc_dir / "plot.png", dpi=300, bbox_inches="tight", pad_inches=0.05)
plt.show()
